# Lab 05 — Sampling, Confidence Intervals, and Hypothesis Testing

This lab follows the **Week 5 lecture** and uses **two real datasets**.

1. **UCI Red Wine Quality** — pH, alcohol, and quality measurements.
2. **Avocado Prices** — avocado prices and market observations.

### Topics
- Sampling distributions and the Central Limit Theorem (CLT)
- Point estimates, standard error, and confidence intervals
- Hypothesis testing and p-values
- Total Variation Distance (TVD)
- Comparing groups and permutation testing

### Grading
- **Autograded: 75 points**
- **Manual: 25 points**
- **Total: 100 points**

> **Note:** Autograded questions include both **public and hidden tests**. Passing the public tests does not guarantee full credit, so make sure your code follows all question requirements.


## Lab Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import zipfile

rng_seed = 42
rng = np.random.default_rng(rng_seed)

import otter
grader = otter.Notebook()

## Real Dataset — UCI Red Wine Quality

We will use the same type of real-world wine data used in the earlier lab. Each row represents a red wine sample, with physicochemical measurements such as **pH, alcohol, fixed acidity**, and a sensory **quality** score.

The dataset is loaded directly from the UCI Machine Learning Repository.


In [ ]:
wine = pd.read_csv('data/winequality-red.csv')
print("Dataset shape:", wine.shape)
wine.head()

Before beginning, examine the available variables and a few summary statistics.


In [ ]:
wine[["pH", "alcohol", "fixed acidity", "quality"]].describe()

## Second Real Dataset — Avocado Prices

The earlier SP26 lab also introduced an **avocado dataset**. We will use it in the later exercises so that this lab works with more than one real dataset.

Important variables include:

- `AveragePrice` — average avocado price.
- `type` — conventional or organic.
- `region` — market/region.
- `year` — observation year.

The data are loaded from a public copy of the avocado dataset.


In [ ]:
# Example code of how to load in data stored within a zip-file
# import zipfile  # already imported above 
zf = zipfile.ZipFile('data/avocado.zip')
avocado = pd.read_csv(zf.open('avocado.csv'), index_col = 0)
avocado.head()

In [ ]:
avocado[["AveragePrice", "year"]].describe()

# Part A — Sampling Distributions and CLT

For this section, treat the observed `wine["pH"]` values as the population available to us. We will repeatedly sample from these values to study how sample means behave.


### Before Exercise 1: Population vs. Sample

A **population mean** describes the center of all observations in the population, while a **sample mean** describes the center of a selected sample.

For example:

```python
population_mean = data.mean()
sample = rng.choice(data, size=20, replace=True)
sample_mean = sample.mean()
```

Because a sample contains only some observations, its mean will usually differ slightly from the population mean.


## Exercise 1 — Wine pH: Population Mean and One Sample [10 points, autograded]

Using the real wine `pH` data:

1. Compute the mean of all `wine["pH"]` values and store it in `population_mean`.
2. Using `np.random.default_rng(101)`, select **30 pH observations with replacement** and store their mean in `one_sample_mean`.

Do not hard-code either mean.


In [ ]:

population_mean = ...

q1_rng = np.random.default_rng(101)
sample = q1_rng.choice(wine["pH"].to_numpy(), size=30, replace=True)
one_sample_mean = ...

In [ ]:
grader.check("q1")

### Before Exercise 2: Sampling Distribution

A single sample gives one estimate. Repeating the sampling process many times gives many sample means.

```python
means = np.zeros(100)

for i in range(100):
    sample = rng.choice(data, size=20, replace=True)
    means[i] = sample.mean()
```

The values stored in `means` form a **sampling distribution of the sample mean**.


## Exercise 2 — Repeated Wine pH Sample Means [15 points, autograded]

Draw **500 samples**, each containing **30 wine pH values**, with replacement.

The loop is provided. Complete the missing calculations so that:

- `means[i]` stores the mean pH of each sample.
- `mean_of_sample_means` stores the mean of all 500 sample means.

Use the provided `q2_rng` so your results are reproducible.


In [ ]:

q2_rng = np.random.default_rng(202)
N_tests = 500
n = 30
means = np.zeros(N_tests)

for i in range(N_tests):
    sampled_values = q2_rng.choice(wine["pH"].to_numpy(), size=n, replace=True)
    means[i] = ...

mean_of_sample_means = ...

In [ ]:
grader.check("q2")

### Before Exercise 3: Central Limit Theorem

The **Central Limit Theorem (CLT)** tells us that the distribution of sample means tends to be approximately normal as the sample size becomes sufficiently large.

Compare:

- the distribution of the original wine pH observations, and
- the distribution of the 500 sample means from Exercise 2.

Look particularly at their **shape and spread**.


<!-- BEGIN QUESTION -->

## Exercise 3 — Visualize the Wine pH Sampling Distribution [10 points, manual]

Create:

1. A histogram of the original `wine["pH"]` values.
2. A histogram of the sample means stored in `means`.

Then write **2–3 sentences** comparing the two distributions and relating your observation to the **Central Limit Theorem**.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(wine["pH"], bins=20)
axes[0].set_title("Original Wine pH")
axes[0].set_xlabel("pH")
axes[0].set_ylabel("Count")

axes[1].hist(means, bins=20)
axes[1].set_title("Distribution of Sample Mean pH")
axes[1].set_xlabel("Sample Mean pH")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

**ANSWER HERE:**


<!-- END QUESTION -->

# Part B — Confidence Intervals Using Wine Data

A **point estimate** gives a single estimate of a population parameter. A confidence interval adds information about uncertainty around that estimate.

The standard error of a sample mean is

$$
SE = \frac{s}{\sqrt{n}}
$$

where $s$ is the sample standard deviation and $n$ is the sample size.

A 95% normal confidence interval can be computed with:

```python
stats.norm.interval(0.95, loc=sample_mean, scale=standard_error)
```


## Exercise 4 — 95% CI for Mean Wine pH [15 points, autograded]

Use the **first 64 wine pH observations** as a sample.

1. Store their mean in `sample_pH_mean`.
2. Store their sample standard deviation in `sample_pH_std` using Pandas' default standard deviation.
3. Compute `standard_error = sample_pH_std / sqrt(n)`.
4. Compute the 95% normal confidence interval and store it in `CI95`.

The sample selection is already provided.


In [ ]:

pH_sample = wine["pH"].iloc[:64]
n = len(pH_sample)

sample_pH_mean = ...
sample_pH_std = ...
standard_error = ...
CI95 = stats.norm.interval(..., loc=..., scale=...)

In [ ]:
grader.check("q4")

<!-- BEGIN QUESTION -->

## Exercise 5 — Interpret the Wine pH Confidence Interval [5 points, manual]

In **1–2 sentences**, interpret the 95% confidence interval from Exercise 4 as an estimate of the population mean wine pH.

Remember that a 95% confidence level describes the long-run performance of the interval-building procedure.


**ANSWER HERE:**


<!-- END QUESTION -->

# Part C — Hypothesis Testing and p-values with Wine Quality

Suppose a winery claims that **50% of its red wines are rated quality 6 or higher**.

We can treat this as the null hypothesis:

$$
H_0: p = 0.50
$$

where a “success” means `quality >= 6`.

We will use the **first 50 wines** as our observed sample and compare the observed number of successes with results simulated under the null model.


## Exercise 6 — Wine Quality Hypothesis Test [15 points, autograded]

Using the first **50 wines**:

1. Count how many have `quality >= 6` and store the result in `observed_high_quality`.
2. Under $H_0: p=0.50$, simulate **5,000 samples of 50 wines** using a binomial distribution.
3. Compute a **two-sided simulation p-value** using the statistic

$$
|\text{number of high-quality wines} - 25|.
$$

Store the simulated counts in `simulated_high_quality`, the observed statistic in `observed_quality_stat`, and the p-value in `p_value`.


In [ ]:

wine_sample50 = wine.iloc[:50]
observed_high_quality = ...
observed_quality_stat = ...

q6_rng = np.random.default_rng(606)
simulated_high_quality = q6_rng.binomial(n=50, p=0.5, size=5000)
simulated_quality_stats = ...

p_value = ...

In [ ]:
grader.check("q6")

<!-- BEGIN QUESTION -->

## Exercise 7 — Interpret the Wine Quality p-value [5 points, manual]

Using **α = 0.05**, write **2–3 sentences** explaining:

- what your p-value means under the winery's 50% null model;
- whether you would **reject** or **fail to reject** the null hypothesis.

Do not interpret the p-value as the probability that the null hypothesis is true.


**ANSWER HERE:**


<!-- END QUESTION -->

# Part D — Total Variation Distance (TVD) with Avocado Data

TVD measures the overall difference between two categorical distributions:

$$
TVD(A,B)=\frac{1}{2}\sum_i |A_i-B_i|.
$$

For the avocado data, we will compare the distribution of **price categories** for:

- conventional avocados, and
- organic avocados.

We define three price categories using `AveragePrice`:

- **Low:** price < 1.00
- **Medium:** 1.00 ≤ price < 1.50
- **High:** price ≥ 1.50


In [ ]:
price_bins = [0, 1.00, 1.50, np.inf]
price_labels = ["Low", "Medium", "High"]

avocado = avocado.copy()
avocado["PriceCategory"] = pd.cut(
    avocado["AveragePrice"],
    bins=price_bins,
    labels=price_labels,
    right=False
)

conventional = avocado[avocado["type"] == "conventional"]
organic = avocado[avocado["type"] == "organic"]

conventional_price_dist = (
    conventional["PriceCategory"].value_counts(normalize=True)
    .reindex(price_labels, fill_value=0)
    .to_numpy()
)

organic_price_dist = (
    organic["PriceCategory"].value_counts(normalize=True)
    .reindex(price_labels, fill_value=0)
    .to_numpy()
)

pd.DataFrame({
    "PriceCategory": price_labels,
    "Conventional": conventional_price_dist,
    "Organic": organic_price_dist
})

## Exercise 8 — TVD Between Avocado Price Distributions [10 points, autograded]

Using `conventional_price_dist` and `organic_price_dist`:

1. Compute the absolute category-by-category differences and store them in `absolute_differences`.
2. Compute the Total Variation Distance and store it in `tvd`.

Use:

$$
TVD=\frac{1}{2}\sum |A_i-B_i|.
$$


In [ ]:

absolute_differences = ...
tvd = ...

In [ ]:
grader.check("q8")

# Part E — Comparing Avocado Groups and Permutation Testing

We will compare **AveragePrice** between two avocado groups:

- **Group A:** conventional avocados
- **Group B:** organic avocados

Our observed statistic is:

$$
\text{Mean Price}(B)-\text{Mean Price}(A).
$$

A permutation test then asks whether a difference this large would be unusual if avocado type were unrelated to `AveragePrice`.


In [ ]:
group_A = avocado.loc[avocado["type"] == "conventional", "AveragePrice"].dropna().to_numpy()
group_B = avocado.loc[avocado["type"] == "organic", "AveragePrice"].dropna().to_numpy()

print("Conventional group size:", len(group_A))
print("Organic group size:", len(group_B))

## Exercise 9 — Difference in Mean Avocado Price [10 points, autograded]

Compute:

- `mean_A`: mean `AveragePrice` for conventional avocados;
- `mean_B`: mean `AveragePrice` for organic avocados;
- `observed_difference`: **mean(B) − mean(A)**.

A positive difference means that organic avocados have a higher average price in this dataset.


In [ ]:

mean_A = ...
mean_B = ...
observed_difference = ...

In [ ]:
grader.check("q9")

### Before Exercise 10: Permutation Testing

Under the null hypothesis, the observed avocado prices are not systematically associated with avocado type. We can therefore shuffle the combined `AveragePrice` values and repeatedly split them into groups of the original sizes.

The resulting differences form a **permutation distribution**.

The code below performs the shuffling for you; your task is to interpret the result.


<!-- BEGIN QUESTION -->

## Exercise 10 — Interpret the Avocado Permutation Test [5 points, manual]

Run the provided permutation-test code.

Then write **2–3 sentences** explaining:

1. Why shuffling `AveragePrice` values between the two group sizes represents the null hypothesis.
2. Whether the observed difference appears unusual compared with the permutation distribution.

Use the displayed permutation p-value to support your interpretation.


In [ ]:
q10_rng = np.random.default_rng(1010)

combined = np.concatenate([group_A, group_B])
n_A = len(group_A)
permuted_differences = np.zeros(1000)

for i in range(1000):
    shuffled = q10_rng.permutation(combined)
    shuffled_A = shuffled[:n_A]
    shuffled_B = shuffled[n_A:]
    permuted_differences[i] = shuffled_B.mean() - shuffled_A.mean()

permutation_p_value = (
    np.abs(permuted_differences) >= abs(observed_difference)
).mean()

plt.hist(permuted_differences, bins=25)
plt.axvline(observed_difference, linestyle="--")
plt.xlabel("Difference in Mean AveragePrice (Organic - Conventional)")
plt.ylabel("Count")
plt.title("Avocado Price Permutation Distribution")
plt.show()

print("Observed difference:", observed_difference)
print("Permutation p-value:", permutation_p_value)

**ANSWER HERE:**


<!-- END QUESTION -->

## 🎉 Congratulations! You Have Completed Lab 05

## Submission Instructions

Before submitting:

- Run all notebook cells in order.
- Make sure both real datasets load successfully.
- Confirm that all public tests pass.
- Complete all manually graded plots and written responses.
- Save the notebook before submitting.

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)